In [ ]:
# Improved imports and data loading
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load processed outputs generated by compute_co2.py
# If filenames differ, update the paths below.
df = pd.read_csv('co2_per_product.csv')
summary = pd.read_csv('summary_by_product_type_ordered.csv', index_col=0)
top5 = pd.read_csv('top5_by_product_type.csv', index_col=0)
bottom5 = pd.read_csv('bottom5_by_product_type.csv', index_col=0)

# Basic overview
print('Rows total:', len(df))
print('Rows with CO2 computed:', df['co2_per_product'].notna().sum())
print('Rows missing CO2 (NaN):', df['co2_per_product'].isna().sum())

# Ensure numeric column
df['co2_per_product'] = pd.to_numeric(df['co2_per_product'], errors='coerce')


In [ ]:
# Show ordered summary (most sustainable -> least).
# The CSV is already ordered ascending by mean (lowest = most sustainable).
display(summary.head(12))


In [ ]:
# Annotated horizontal bar for the top categories (most sustainable -> least)
sns.set(style='whitegrid')
sel = summary.head(40)  # first 40 for readability
plt.figure(figsize=(11, max(5, len(sel)*0.25)))
ax = sns.barplot(x='mean', y=sel.index, data=sel.reset_index(), palette='viridis')
ax.set_xlabel('Mean CO2 per product (kg CO2e / kg)')
ax.set_ylabel(summary.index.name or 'product_type')
ax.set_title('Mean CO2 per product by product type (most sustainable → least)')
for p in ax.patches:
    x = p.get_width()
    if np.isfinite(x):
        ax.text(x + 0.02 * summary['mean'].max(), p.get_y() + p.get_height() / 2, f"{x:.2f}", va='center')
plt.tight_layout()
plt.show()


In [ ]:
# Top 5 worst offenders (show table and a small explanatory note)
display(top5[['mean','std','count']])

print('\nNote: High mean + high std indicates both high impact and high variability across items in that category.')


In [ ]:
# Bottom 5 (most sustainable) — categories with lowest mean CO2
display(bottom5[['mean','std','count']])

print('\nNote: Very low means can indicate either genuinely low-impact items or missing/limited percentage reporting. Check materials for those categories before concluding.')


## Short textual summary (copyable for slides)

- Total products in dataset: **replace with the number printed above**.
- Products used for CO₂ calculation (explicit material percentages): **replace with computed count above**.
- Products skipped (no percentage info): **replace with computed count above**.

Suggested slide text (paste and update numbers):

"Based on product-level material compositions, the mean cradle-to-gate CO₂ footprint per product varies across categories. The highest average impacts are observed in knitwear and heavy outerwear due to wool and synthetic blends, while accessory categories and products with high recycled content show lower averages."

Update the numbers and add 1–2 SKU examples from the code cell below when you paste into slides.


In [ ]:
# Quick helper: concrete numbers and example SKUs to paste into slides
print('Total products:', len(df))
print('Computed CO2 (with percentages):', df['co2_per_product'].notna().sum())
print('Skipped (no percentages):', df['co2_per_product'].isna().sum())

print('\nTop 5 worst offenders (mean CO2, count):')
display(top5[['mean','count']])

print('\nTop 5 most sustainable (mean CO2, count):')
display(bottom5[['mean','count']])

# Show a couple of example SKUs from the top 3 worst categories
worst_cats = top5.index.tolist()[:3]
for c in worst_cats:
    print(f'Examples from category: {c}')
    keycol = summary.index.name or 'product_type'
    # robust access: try group by the grouping column name used by compute_co2
    # Try common column names
    candidates = ['product_type_name','product_group_name','department_name','mainCatCode']
    group_col = None
    for cand in candidates:
        if cand in df.columns:
            group_col = cand
            break
    if group_col is None:
        print('  (no grouping column found in data)')
        break
    ex = df[df[group_col]==c].dropna(subset=['co2_per_product']).head(3)[['productId','productName','co2_per_product']]
    if not ex.empty:
        display(ex)
    else:
        print('  (no example with computed CO2)')
